In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../outputs/merged_data.csv')

graph_df = df[['TransactionID', 'isFraud', 'card1', 'card2', 'addr1', 'P_emaildomain']].copy()

print("Shape:", graph_df.shape)
print("\nSample:")
print(graph_df.head())

Shape: (590540, 6)

Sample:
   TransactionID  isFraud  card1  card2  addr1 P_emaildomain
0        2987000        0  13926    NaN  315.0           NaN
1        2987001        0   2755  404.0  325.0     gmail.com
2        2987002        0   4663  490.0  330.0   outlook.com
3        2987003        0  18132  567.0  476.0     yahoo.com
4        2987004        0   4497  514.0  420.0     gmail.com


In [2]:
print("Unique card1 values:", graph_df['card1'].nunique())
print("Unique card2 values:", graph_df['card2'].nunique())
print("Unique addr1 values:", graph_df['addr1'].nunique())
print("Unique P_emaildomain values:", graph_df['P_emaildomain'].nunique())

Unique card1 values: 13553
Unique card2 values: 500
Unique addr1 values: 332
Unique P_emaildomain values: 59


In [4]:
import networkx as nx

G = nx.Graph()

# Fill nulls with placeholder
graph_df['card1'] = graph_df['card1'].fillna(-1).astype(str)
graph_df['card2'] = graph_df['card2'].fillna(-1).astype(str)
graph_df['addr1'] = graph_df['addr1'].fillna(-1).astype(str)
graph_df['P_emaildomain'] = graph_df['P_emaildomain'].fillna('unknown')

# Add prefixes to avoid node collisions across entity types
graph_df['card1_node'] = 'card1_' + graph_df['card1']
graph_df['card2_node'] = 'card2_' + graph_df['card2']
graph_df['addr1_node'] = 'addr1_' + graph_df['addr1']
graph_df['email_node'] = 'email_' + graph_df['P_emaildomain']

# Build edges — connect card1 to each attribute it appears with
for _, row in graph_df.iterrows():
    G.add_edge(row['card1_node'], row['card2_node'])
    G.add_edge(row['card1_node'], row['addr1_node'])
    G.add_edge(row['card1_node'], row['email_node'])

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 14447
Edges: 97979


In [5]:
print("Computing degree centrality...")
degree_centrality = nx.degree_centrality(G)

print("Computing clustering coefficient...")
clustering = nx.clustering(G)

print("Done")

Computing degree centrality...
Computing clustering coefficient...
Done


In [6]:
graph_df['degree_centrality'] = graph_df['card1_node'].map(degree_centrality)
graph_df['clustering_coef'] = graph_df['card1_node'].map(clustering)

# Raw degree (number of direct connections)
degree_dict = dict(G.degree())
graph_df['node_degree'] = graph_df['card1_node'].map(degree_dict)

print("Sample graph features:")
print(graph_df[['TransactionID', 'isFraud', 'degree_centrality', 
                 'clustering_coef', 'node_degree']].head(10))

Sample graph features:
   TransactionID  isFraud  degree_centrality  clustering_coef  node_degree
0        2987000        0           0.001800                0           26
1        2987001        0           0.005053                0           73
2        2987002        0           0.004292                0           62
3        2987003        0           0.004500                0           65
4        2987004        0           0.000900                0           13
5        2987005        0           0.000485                0            7
6        2987006        0           0.002977                0           43
7        2987007        0           0.004846                0           70
8        2987008        0           0.006853                0           99
9        2987009        0           0.003807                0           55


In [7]:
print("Average graph features by fraud label:")
print(graph_df.groupby('isFraud')[['degree_centrality', 'node_degree']].mean())

Average graph features by fraud label:
         degree_centrality  node_degree
isFraud                                
0                 0.003287    47.487972
1                 0.003237    46.759570


In [8]:
# How many unique card1s share the same addr1
addr_card_count = graph_df.groupby('addr1')['card1'].nunique()
graph_df['addr_shared_cards'] = graph_df['addr1'].map(addr_card_count)

# How many unique card1s share the same email domain
email_card_count = graph_df.groupby('P_emaildomain')['card1'].nunique()
graph_df['email_shared_cards'] = graph_df['P_emaildomain'].map(email_card_count)

# How many unique card1s share the same card2
card2_card_count = graph_df.groupby('card2')['card1'].nunique()
graph_df['card2_shared_cards'] = graph_df['card2'].map(card2_card_count)

print("Average shared entity counts by fraud label:")
print(graph_df.groupby('isFraud')[['addr_shared_cards', 
                                    'email_shared_cards', 
                                    'card2_shared_cards']].mean())

Average shared entity counts by fraud label:
         addr_shared_cards  email_shared_cards  card2_shared_cards
isFraud                                                           
0              1320.885867         6027.544045          518.853191
1              1630.841601         6388.144413          358.762087


In [9]:
graph_features = graph_df[[
    'TransactionID',
    'degree_centrality',
    'node_degree',
    'addr_shared_cards',
    'email_shared_cards',
    'card2_shared_cards'
]].copy()

print("Graph features shape:", graph_features.shape)
graph_features.to_csv('../outputs/graph_features.csv', index=False)
print("Saved to outputs/graph_features.csv")

Graph features shape: (590540, 6)
Saved to outputs/graph_features.csv


## Graph Module Results

Built a transaction network with 14,447 nodes and 97,979 edges connecting:
- `card1` nodes (individual cards) to `card2`, `addr1`, and `P_emaildomain` nodes

**Node-level features (card1 centrality):**
- `degree_centrality` and `node_degree` showed minimal difference between fraud/legit
- Card-level connectivity alone is not a strong fraud signal

**Shared entity features (the real signal):**
- `addr_shared_cards` — fraud transactions come from addresses shared by more unique cards 
  (1,630 vs 1,320 avg) — fraudsters cluster around fewer physical addresses
- `email_shared_cards` — fraud transactions use email domains shared by more cards (6,388 vs 6,027)
- `card2_shared_cards` — fraud transactions have LOWER card2 sharing (358 vs 518) — 
  fraudsters avoid reusing card2 values, an evasion pattern

**Key insight:** Fraud rings share addresses but diversify card numbers — 
the graph reveals structural patterns invisible to tabular features alone.